<a href="https://colab.research.google.com/github/yhou151209/541_project/blob/main/541.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 28.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []
    data["preferences"].append(preference)


def is_busy_shift(day: str, shift: str) -> bool:
    if shift == "evening":
        return True
    if day in {"Saturday", "Sunday"} and shift in {"morning", "evening"}:
        return True
    return False


def get_day_order() -> Dict[str, int]:
    return {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }


def get_sorted_days(days: List[str]) -> List[str]:
    day_order = get_day_order()
    return sorted(set(days), key=lambda d: day_order.get(d, 999))


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = get_sorted_days([r["day"] for r in requirements])
    all_shifts = sorted({r["shift"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    penalty_terms: List[cp_model.LinearExpr] = []

    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]
        penalty = int(pref.get("penalty", 1))

        if pref_type == "avoid_shift":
            target_day = pref.get("day")
            target_shift = pref.get("shift")

            for role in all_roles:
                for day in all_days:
                    for shift in all_shifts:
                        day_match = (target_day is None) or (day == target_day)
                        shift_match = (target_shift is None) or (shift == target_shift)

                        if day_match and shift_match:
                            key = (emp_id, day, shift, role)
                            if key in x:
                                penalty_terms.append(x[key] * penalty)

        elif pref_type == "avoid_back_to_back":
            # Same-day back-to-back: morning + evening on the same day
            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = model.NewBoolVar(f"morning_assigned_{emp_id}_{day}_{i}")
                evening_assigned = model.NewBoolVar(f"evening_assigned_{emp_id}_{day}_{i}")
                same_day_back_to_back = model.NewBoolVar(f"same_day_back_to_back_{emp_id}_{day}_{i}")

                model.Add(sum(morning_vars) >= 1).OnlyEnforceIf(morning_assigned)
                model.Add(sum(morning_vars) == 0).OnlyEnforceIf(morning_assigned.Not())

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(same_day_back_to_back)
                model.AddBoolOr(
                    [morning_assigned.Not(), evening_assigned.Not()]
                ).OnlyEnforceIf(same_day_back_to_back.Not())

                penalty_terms.append(same_day_back_to_back * penalty)

            # Cross-day back-to-back: evening on day d + morning on day d+1
            for day_index in range(len(all_days) - 1):
                day = all_days[day_index]
                next_day = all_days[day_index + 1]

                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]
                next_morning_vars = [
                    x[(emp_id, next_day, "morning", role)]
                    for role in all_roles
                    if (emp_id, next_day, "morning", role) in x
                ]

                if not evening_vars or not next_morning_vars:
                    continue

                evening_assigned = model.NewBoolVar(
                    f"cross_evening_assigned_{emp_id}_{day}_{i}"
                )
                next_morning_assigned = model.NewBoolVar(
                    f"cross_next_morning_assigned_{emp_id}_{next_day}_{i}"
                )
                cross_day_back_to_back = model.NewBoolVar(
                    f"cross_day_back_to_back_{emp_id}_{day}_to_{next_day}_{i}"
                )

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.Add(sum(next_morning_vars) >= 1).OnlyEnforceIf(next_morning_assigned)
                model.Add(sum(next_morning_vars) == 0).OnlyEnforceIf(next_morning_assigned.Not())

                model.AddBoolAnd(
                    [evening_assigned, next_morning_assigned]
                ).OnlyEnforceIf(cross_day_back_to_back)
                model.AddBoolOr(
                    [evening_assigned.Not(), next_morning_assigned.Not()]
                ).OnlyEnforceIf(cross_day_back_to_back.Not())

                penalty_terms.append(cross_day_back_to_back * penalty)

    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    total_penalty = model.NewIntVar(0, 50000, "total_penalty")
    all_penalties = penalty_terms + busy_shift_penalty_terms

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    day_order = {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }

    shift_order = {
        "morning": 1,
        "evening": 2,
    }

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def get_removed_and_added_assignments(
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    old_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in old_schedule
    }
    new_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in new_schedule
    }

    removed = old_set - new_set
    added = new_set - old_set

    removed_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(removed)
    ]

    added_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(added)
    ]

    return removed_rows, added_rows


def generate_explanation(
    data_before: Dict[str, Any],
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> str:
    change_type = change.get("type", "").strip()
    removed_rows, added_rows = get_removed_and_added_assignments(old_schedule, new_schedule)
    lines: List[str] = []

    if change_type == "unavailable":
        employee_name = change["employee_name"]
        day = change["day"]
        shift = change["shift"]

        lines.append(
            f"{employee_name} was removed because they are unavailable for {day} {shift}."
        )

        replacement_rows = [
            row for row in added_rows
            if row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower()
        ]

        if replacement_rows:
            for row in replacement_rows:
                emp_info = next(
                    (s for s in data_before["staff"] if s["name"].lower() == row["employee_name"].lower()),
                    None
                )
                if emp_info and emp_info["employment_type"] == "full-time" and is_busy_shift(row["day"], row["shift"]):
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available, qualified as a {row['role']}, and full-time staff are preferred on busy shifts."
                    )
                else:
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available and qualified as a {row['role']}."
                    )

        if not replacement_rows:
            lines.append(
                "The schedule was re-optimized to keep the shift covered with available qualified staff."
            )

    elif change_type == "direct_swap":
        employee_name_1 = change["employee_name_1"]
        day_1 = change["day_1"]
        shift_1 = change["shift_1"]
        employee_name_2 = change["employee_name_2"]
        day_2 = change["day_2"]
        shift_2 = change["shift_2"]

        lines.append(
            f"A direct swap was performed between {employee_name_1} ({day_1} {shift_1}) and {employee_name_2} ({day_2} {shift_2})."
        )
        lines.append(
            "The swap was accepted because both employees were assigned to those shifts and remained qualified for the exchanged roles."
        )

    elif change_type == "avoid_back_to_back":
        employee_name = change["employee_name"]
        day_order = get_day_order()
        all_days = sorted(
            {row["day"] for row in old_schedule + new_schedule},
            key=lambda d: day_order.get(d, 999)
        )

        same_day_before_count = 0
        same_day_after_count = 0
        cross_day_before_count = 0
        cross_day_after_count = 0

        for day in all_days:
            old_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in old_schedule
            )
            old_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in old_schedule
            )
            new_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in new_schedule
            )
            new_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in new_schedule
            )

            if old_morning and old_evening:
                same_day_before_count += 1
            if new_morning and new_evening:
                same_day_after_count += 1

        for idx in range(len(all_days) - 1):
            day = all_days[idx]
            next_day = all_days[idx + 1]

            old_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in old_schedule
            )
            old_next_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == next_day and r["shift"] == "morning"
                for r in old_schedule
            )
            new_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in new_schedule
            )
            new_next_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == next_day and r["shift"] == "morning"
                for r in new_schedule
            )

            if old_evening and old_next_morning:
                cross_day_before_count += 1
            if new_evening and new_next_morning:
                cross_day_after_count += 1

        lines.append(
            f"A preference was added to avoid assigning {employee_name} to back-to-back shifts."
        )
        lines.append(
            f"Same-day back-to-back assignments (morning + evening) changed from {same_day_before_count} to {same_day_after_count}."
        )
        lines.append(
            f"Cross-day back-to-back assignments (evening + next-day morning) changed from {cross_day_before_count} to {cross_day_after_count}."
        )
        lines.append(
            "The schedule was re-optimized to reduce both same-day and cross-day back-to-back assignments when possible."
        )

    elif change_type == "avoid_shift":
        employee_name = change["employee_name"]
        target_day = change.get("day")
        target_shift = change.get("shift")

        def match_target(row: Dict[str, str]) -> bool:
            if row["employee_name"].lower() != employee_name.lower():
                return False
            if target_day is not None and row["day"] != target_day:
                return False
            if target_shift is not None and row["shift"] != target_shift:
                return False
            return True

        before_count = sum(1 for row in old_schedule if match_target(row))
        after_count = sum(1 for row in new_schedule if match_target(row))

        if target_day and target_shift:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to {target_day} {target_shift} shifts."
            )
            lines.append(
                f"Assignments for that time slot changed from {before_count} to {after_count}."
            )
        elif target_day:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} on {target_day}."
            )
            lines.append(
                f"Assignments on {target_day} changed from {before_count} to {after_count}."
            )
        elif target_shift:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to {target_shift} shifts."
            )
            lines.append(
                f"{target_shift.capitalize()} assignments for {employee_name} changed from {before_count} to {after_count}."
            )
        else:
            lines.append(
                f"A general avoidance preference was added for {employee_name}."
            )

        lines.append(
            "The schedule was re-optimized to reduce those assignments when possible."
        )

    else:
        lines.append("The schedule was updated and re-optimized based on the requested change.")

    full_time_staff = {
        s["name"] for s in data_before["staff"] if s["employment_type"] == "full-time"
    }

    busy_added = [row for row in added_rows if is_busy_shift(row["day"], row["shift"])]
    if busy_added:
        full_time_busy_added = [row for row in busy_added if row["employee_name"] in full_time_staff]
        if full_time_busy_added:
            names = ", ".join(sorted({row["employee_name"] for row in full_time_busy_added}))
            lines.append(
                f"Busy shifts prefer full-time staff when possible, which influenced assignments such as {names}."
            )
        else:
            lines.append(
                "Busy shifts prefer full-time staff when possible, but availability and skill constraints still had to be satisfied."
            )

    if removed_rows or added_rows:
        lines.append("Other assignments were kept unchanged when possible.")

    return "\n".join(lines)


def print_explanation(explanation: str) -> None:
    print("===== EXPLANATION =====")
    print(explanation)
    print("=======================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")
        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type == "avoid_back_to_back":
        employee_name = change["employee_name"].strip()
        penalty = int(change.get("penalty", 5))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_back_to_back",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_shift":
        employee_name = change["employee_name"].strip()
        target_day = change.get("day")
        target_shift = change.get("shift")
        penalty = int(change.get("penalty", 3))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        pref = {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "penalty": penalty,
        }
        if target_day:
            pref["day"] = target_day.strip()
        if target_shift:
            pref["shift"] = target_shift.strip()

        add_preference(updated_data, pref)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'direct_swap', 'avoid_back_to_back', and 'avoid_shift'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")
    print("3. avoid_back_to_back")
    print("4. avoid_shift")

    while True:
        choice = input("Enter 1, 2, 3, or 4: ").strip()
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = input("Employee 1 current day: ").strip()
        shift_1 = input("Employee 1 current shift: ").strip()

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = input("Employee 2 current day: ").strip()
        shift_2 = input("Employee 2 current shift: ").strip()

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        penalty_text = input("Penalty weight (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5

        return {
            "type": "avoid_back_to_back",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    print("Choose avoid type:")
    print("1. avoid a shift type (e.g. evening)")
    print("2. avoid a day (e.g. Tuesday)")
    print("3. avoid a specific day and shift (e.g. Tuesday evening)")

    while True:
        sub_choice = input("Enter 1, 2, or 3: ").strip()
        if sub_choice in {"1", "2", "3"}:
            break
        print("Please enter 1, 2, or 3.")

    if sub_choice == "1":
        shift = input("Shift to avoid (e.g. evening): ").strip()
        penalty_text = input("Penalty weight (default 3): ").strip()
        penalty = int(penalty_text) if penalty_text else 3

        return {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "shift": shift,
            "penalty": penalty,
        }

    if sub_choice == "2":
        day = input("Day to avoid (e.g. Tuesday): ").strip()
        penalty_text = input("Penalty weight (default 3): ").strip()
        penalty = int(penalty_text) if penalty_text else 3

        return {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "day": day,
            "penalty": penalty,
        }

    day = input("Day to avoid (e.g. Tuesday): ").strip()
    shift = input("Shift to avoid (e.g. evening): ").strip()
    penalty_text = input("Penalty weight (default 3): ").strip()
    penalty = int(penalty_text) if penalty_text else 3

    return {
        "type": "avoid_shift",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
        "penalty": penalty,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")
    print_preferences(data)

    while True:
        should_update = ask_yes_no("Do you want to enter an update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_change_request()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            explanation = generate_explanation(data, schedule, updated_schedule, change_request)
            print_explanation(explanation)

            print_preferences(updated_data)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")
            print_preferences(data)


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Saturday - mor

In [5]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")


def clone_data(data: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }


def clone_schedule(schedule: List[Dict[str, str]]) -> List[Dict[str, str]]:
    return [dict(row) for row in schedule]


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []
    data["preferences"].append(preference)


def is_busy_shift(day: str, shift: str) -> bool:
    if shift == "evening":
        return True
    if day in {"Saturday", "Sunday"} and shift in {"morning", "evening"}:
        return True
    return False


def get_day_order() -> Dict[str, int]:
    return {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }


def get_sorted_days(days: List[str]) -> List[str]:
    day_order = get_day_order()
    return sorted(set(days), key=lambda d: day_order.get(d, 999))


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = get_sorted_days([r["day"] for r in requirements])
    all_shifts = sorted({r["shift"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    penalty_terms: List[cp_model.LinearExpr] = []

    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]
        penalty = int(pref.get("penalty", 1))

        if pref_type == "avoid_shift":
            target_day = pref.get("day")
            target_shift = pref.get("shift")

            for role in all_roles:
                for day in all_days:
                    for shift in all_shifts:
                        day_match = (target_day is None) or (day == target_day)
                        shift_match = (target_shift is None) or (shift == target_shift)

                        if day_match and shift_match:
                            key = (emp_id, day, shift, role)
                            if key in x:
                                penalty_terms.append(x[key] * penalty)

        elif pref_type == "avoid_back_to_back":
            # same-day: morning + evening
            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = model.NewBoolVar(f"morning_assigned_{emp_id}_{day}_{i}")
                evening_assigned = model.NewBoolVar(f"evening_assigned_{emp_id}_{day}_{i}")
                same_day_back_to_back = model.NewBoolVar(f"same_day_back_to_back_{emp_id}_{day}_{i}")

                model.Add(sum(morning_vars) >= 1).OnlyEnforceIf(morning_assigned)
                model.Add(sum(morning_vars) == 0).OnlyEnforceIf(morning_assigned.Not())

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(same_day_back_to_back)
                model.AddBoolOr(
                    [morning_assigned.Not(), evening_assigned.Not()]
                ).OnlyEnforceIf(same_day_back_to_back.Not())

                penalty_terms.append(same_day_back_to_back * penalty)

            # cross-day: evening + next-day morning
            for day_index in range(len(all_days) - 1):
                day = all_days[day_index]
                next_day = all_days[day_index + 1]

                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]
                next_morning_vars = [
                    x[(emp_id, next_day, "morning", role)]
                    for role in all_roles
                    if (emp_id, next_day, "morning", role) in x
                ]

                if not evening_vars or not next_morning_vars:
                    continue

                evening_assigned = model.NewBoolVar(
                    f"cross_evening_assigned_{emp_id}_{day}_{i}"
                )
                next_morning_assigned = model.NewBoolVar(
                    f"cross_next_morning_assigned_{emp_id}_{next_day}_{i}"
                )
                cross_day_back_to_back = model.NewBoolVar(
                    f"cross_day_back_to_back_{emp_id}_{day}_to_{next_day}_{i}"
                )

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.Add(sum(next_morning_vars) >= 1).OnlyEnforceIf(next_morning_assigned)
                model.Add(sum(next_morning_vars) == 0).OnlyEnforceIf(next_morning_assigned.Not())

                model.AddBoolAnd(
                    [evening_assigned, next_morning_assigned]
                ).OnlyEnforceIf(cross_day_back_to_back)
                model.AddBoolOr(
                    [evening_assigned.Not(), next_morning_assigned.Not()]
                ).OnlyEnforceIf(cross_day_back_to_back.Not())

                penalty_terms.append(cross_day_back_to_back * penalty)

    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    total_penalty = model.NewIntVar(0, 50000, "total_penalty")
    all_penalties = penalty_terms + busy_shift_penalty_terms

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    day_order = get_day_order()
    shift_order = {"morning": 1, "evening": 2}
    schedule.sort(
        key=lambda r: (
            day_order.get(r["day"], 999),
            shift_order.get(r["shift"], 999),
            r["role"],
            r["employee_name"],
        )
    )
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def start_next_week(
    current_data: Dict[str, Any],
    current_final_schedule: List[Dict[str, str]],
) -> Tuple[Dict[str, Any], List[Dict[str, str]]]:
    """
    Rule:
    this week's final schedule = next week's initial schedule
    this week's updated data   = next week's starting data
    """
    next_week_data = clone_data(current_data)
    next_week_initial_schedule = clone_schedule(current_final_schedule)
    return next_week_data, next_week_initial_schedule


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    day_order = get_day_order()
    shift_order = {
        "morning": 1,
        "evening": 2,
    }

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def get_removed_and_added_assignments(
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    old_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in old_schedule
    }
    new_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in new_schedule
    }

    removed = old_set - new_set
    added = new_set - old_set

    removed_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(removed)
    ]

    added_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(added)
    ]

    return removed_rows, added_rows


def generate_explanation(
    data_before: Dict[str, Any],
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> str:
    change_type = change.get("type", "").strip()
    removed_rows, added_rows = get_removed_and_added_assignments(old_schedule, new_schedule)
    lines: List[str] = []

    if change_type == "unavailable":
        employee_name = change["employee_name"]
        day = change["day"]
        shift = change["shift"]

        lines.append(
            f"{employee_name} was removed because they are unavailable for {day} {shift}."
        )

        replacement_rows = [
            row for row in added_rows
            if row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower()
        ]

        if replacement_rows:
            for row in replacement_rows:
                emp_info = next(
                    (s for s in data_before["staff"] if s["name"].lower() == row["employee_name"].lower()),
                    None
                )
                if emp_info and emp_info["employment_type"] == "full-time" and is_busy_shift(row["day"], row["shift"]):
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available, qualified as a {row['role']}, and full-time staff are preferred on busy shifts."
                    )
                else:
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available and qualified as a {row['role']}."
                    )

        if not replacement_rows:
            lines.append(
                "The schedule was re-optimized to keep the shift covered with available qualified staff."
            )

    elif change_type == "direct_swap":
        employee_name_1 = change["employee_name_1"]
        day_1 = change["day_1"]
        shift_1 = change["shift_1"]
        employee_name_2 = change["employee_name_2"]
        day_2 = change["day_2"]
        shift_2 = change["shift_2"]

        lines.append(
            f"A direct swap was performed between {employee_name_1} ({day_1} {shift_1}) and {employee_name_2} ({day_2} {shift_2})."
        )
        lines.append(
            "The swap was accepted because both employees were assigned to those shifts and remained qualified for the exchanged roles."
        )

    elif change_type == "avoid_back_to_back":
        employee_name = change["employee_name"]
        day_order = get_day_order()
        all_days = sorted(
            {row["day"] for row in old_schedule + new_schedule},
            key=lambda d: day_order.get(d, 999)
        )

        same_day_before_count = 0
        same_day_after_count = 0
        cross_day_before_count = 0
        cross_day_after_count = 0

        for day in all_days:
            old_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in old_schedule
            )
            old_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in old_schedule
            )
            new_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in new_schedule
            )
            new_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in new_schedule
            )

            if old_morning and old_evening:
                same_day_before_count += 1
            if new_morning and new_evening:
                same_day_after_count += 1

        for idx in range(len(all_days) - 1):
            day = all_days[idx]
            next_day = all_days[idx + 1]

            old_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in old_schedule
            )
            old_next_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == next_day and r["shift"] == "morning"
                for r in old_schedule
            )
            new_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in new_schedule
            )
            new_next_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == next_day and r["shift"] == "morning"
                for r in new_schedule
            )

            if old_evening and old_next_morning:
                cross_day_before_count += 1
            if new_evening and new_next_morning:
                cross_day_after_count += 1

        lines.append(
            f"A preference was added to avoid assigning {employee_name} to back-to-back shifts."
        )
        lines.append(
            f"Same-day back-to-back assignments (morning + evening) changed from {same_day_before_count} to {same_day_after_count}."
        )
        lines.append(
            f"Cross-day back-to-back assignments (evening + next-day morning) changed from {cross_day_before_count} to {cross_day_after_count}."
        )
        lines.append(
            "The schedule was re-optimized to reduce both same-day and cross-day back-to-back assignments when possible."
        )

    elif change_type == "avoid_shift":
        employee_name = change["employee_name"]
        target_day = change.get("day")
        target_shift = change.get("shift")

        def match_target(row: Dict[str, str]) -> bool:
            if row["employee_name"].lower() != employee_name.lower():
                return False
            if target_day is not None and row["day"] != target_day:
                return False
            if target_shift is not None and row["shift"] != target_shift:
                return False
            return True

        before_count = sum(1 for row in old_schedule if match_target(row))
        after_count = sum(1 for row in new_schedule if match_target(row))

        if target_day and target_shift:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to {target_day} {target_shift} shifts."
            )
            lines.append(
                f"Assignments for that time slot changed from {before_count} to {after_count}."
            )
        elif target_day:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} on {target_day}."
            )
            lines.append(
                f"Assignments on {target_day} changed from {before_count} to {after_count}."
            )
        elif target_shift:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to {target_shift} shifts."
            )
            lines.append(
                f"{target_shift.capitalize()} assignments for {employee_name} changed from {before_count} to {after_count}."
            )
        else:
            lines.append(
                f"A general avoidance preference was added for {employee_name}."
            )

        lines.append(
            "The schedule was re-optimized to reduce those assignments when possible."
        )

    else:
        lines.append("The schedule was updated and re-optimized based on the requested change.")

    full_time_staff = {
        s["name"] for s in data_before["staff"] if s["employment_type"] == "full-time"
    }

    busy_added = [row for row in added_rows if is_busy_shift(row["day"], row["shift"])]
    if busy_added:
        full_time_busy_added = [row for row in busy_added if row["employee_name"] in full_time_staff]
        if full_time_busy_added:
            names = ", ".join(sorted({row["employee_name"] for row in full_time_busy_added}))
            lines.append(
                f"Busy shifts prefer full-time staff when possible, which influenced assignments such as {names}."
            )
        else:
            lines.append(
                "Busy shifts prefer full-time staff when possible, but availability and skill constraints still had to be satisfied."
            )

    if removed_rows or added_rows:
        lines.append("Other assignments were kept unchanged when possible.")

    return "\n".join(lines)


def print_explanation(explanation: str) -> None:
    print("===== EXPLANATION =====")
    print(explanation)
    print("=======================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = clone_data(data)

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")
        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type == "avoid_back_to_back":
        employee_name = change["employee_name"].strip()
        penalty = int(change.get("penalty", 5))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_back_to_back",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_shift":
        employee_name = change["employee_name"].strip()
        target_day = change.get("day")
        target_shift = change.get("shift")
        penalty = int(change.get("penalty", 3))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        pref = {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "penalty": penalty,
        }
        if target_day:
            pref["day"] = target_day.strip()
        if target_shift:
            pref["shift"] = target_shift.strip()

        add_preference(updated_data, pref)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'direct_swap', 'avoid_back_to_back', and 'avoid_shift'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")
    print("3. avoid_back_to_back")
    print("4. avoid_shift")

    while True:
        choice = input("Enter 1, 2, 3, or 4: ").strip()
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = input("Employee 1 current day: ").strip()
        shift_1 = input("Employee 1 current shift: ").strip()

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = input("Employee 2 current day: ").strip()
        shift_2 = input("Employee 2 current shift: ").strip()

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        penalty_text = input("Penalty weight (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5

        return {
            "type": "avoid_back_to_back",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    print("Choose avoid type:")
    print("1. avoid a shift type (e.g. evening)")
    print("2. avoid a day (e.g. Tuesday)")
    print("3. avoid a specific day and shift (e.g. Tuesday evening)")

    while True:
        sub_choice = input("Enter 1, 2, or 3: ").strip()
        if sub_choice in {"1", "2", "3"}:
            break
        print("Please enter 1, 2, or 3.")

    if sub_choice == "1":
        shift = input("Shift to avoid (e.g. evening): ").strip()
        penalty_text = input("Penalty weight (default 3): ").strip()
        penalty = int(penalty_text) if penalty_text else 3

        return {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "shift": shift,
            "penalty": penalty,
        }

    if sub_choice == "2":
        day = input("Day to avoid (e.g. Tuesday): ").strip()
        penalty_text = input("Penalty weight (default 3): ").strip()
        penalty = int(penalty_text) if penalty_text else 3

        return {
            "type": "avoid_shift",
            "employee_name": employee_name,
            "day": day,
            "penalty": penalty,
        }

    day = input("Day to avoid (e.g. Tuesday): ").strip()
    shift = input("Shift to avoid (e.g. evening): ").strip()
    penalty_text = input("Penalty weight (default 3): ").strip()
    penalty = int(penalty_text) if penalty_text else 3

    return {
        "type": "avoid_shift",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
        "penalty": penalty,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule for Week 1...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    week_number = 1

    while True:
        print_schedule(schedule, title=f"WEEK {week_number} INITIAL SCHEDULE")
        print_preferences(data)

        while True:
            should_update = ask_yes_no(f"Do you want to enter an update for Week {week_number}? (y/n): ")
            if not should_update:
                break

            try:
                change_request = prompt_change_request()
                updated_schedule, updated_data = update_schedule(data, schedule, change_request)

                print_schedule(updated_schedule, title=f"WEEK {week_number} UPDATED SCHEDULE")
                compare_schedules(schedule, updated_schedule)

                explanation = generate_explanation(data, schedule, updated_schedule, change_request)
                print_explanation(explanation)

                print_preferences(updated_data)

                data = updated_data
                schedule = updated_schedule

                save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
                if save_changes:
                    save_data_to_json(data, data_file)
                    print(f"Saved updated data to {data_file}")

            except ValueError as e:
                print(f"Error: {e}")

            show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
            if show_current:
                print_schedule(schedule, title=f"WEEK {week_number} CURRENT FINAL SCHEDULE")
                print_preferences(data)

        print_schedule(schedule, title=f"WEEK {week_number} FINAL SCHEDULE")

        move_to_next_week = ask_yes_no(
            f"Do you want to move from Week {week_number} to Week {week_number + 1}? (y/n): "
        )
        if not move_to_next_week:
            print("Done.")
            break

        data, schedule = start_next_week(data, schedule)
        week_number += 1
        print(
            f"Started Week {week_number}. "
            f"By rule, Week {week_number - 1} final schedule is now Week {week_number} initial schedule."
        )


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule for Week 1...

===== WEEK 1 INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E0

In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


MIN_PREFERENCE_PENALTY = -10
MAX_PREFERENCE_PENALTY = 10


def clamp_preference_penalty(value: int) -> int:
    if value < MIN_PREFERENCE_PENALTY or value > MAX_PREFERENCE_PENALTY:
        raise ValueError(
            f"Penalty must be between {MIN_PREFERENCE_PENALTY} and {MAX_PREFERENCE_PENALTY}."
        )
    return value


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")

            if pref["type"] in {"shift_preference", "avoid_shift"}:
                penalty = int(pref.get("penalty", 5))
                clamp_preference_penalty(penalty)

            if pref["type"] in {"back_to_back_preference", "avoid_back_to_back"}:
                penalty = int(pref.get("penalty", 5))
                clamp_preference_penalty(penalty)


def clone_data(data: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }


def clone_schedule(schedule: List[Dict[str, str]]) -> List[Dict[str, str]]:
    return [dict(row) for row in schedule]


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def set_availability_by_pattern(
    data: Dict[str, Any],
    employee_id: str,
    available: bool,
    target_day: str | None = None,
    target_shift: str | None = None,
) -> None:
    if target_day is None and target_shift is None:
        raise ValueError("At least one of target_day or target_shift must be provided.")

    all_day_shift_pairs = {
        (row["day"], row["shift"])
        for row in data["availability"]
        if row["employee_id"] == employee_id
    }
    all_day_shift_pairs.update(
        (req["day"], req["shift"])
        for req in data["shift_requirements"]
    )

    matched_pairs = []
    for day, shift in sorted(all_day_shift_pairs):
        if target_day is not None and day != target_day:
            continue
        if target_shift is not None and shift != target_shift:
            continue
        matched_pairs.append((day, shift))

    if not matched_pairs:
        raise ValueError(
            f"No matching availability slots found for employee_id={employee_id}, "
            f"day={target_day}, shift={target_shift}."
        )

    for day, shift in matched_pairs:
        set_availability(data, employee_id, day, shift, available)


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []
    data["preferences"].append(preference)


def is_busy_shift(day: str, shift: str) -> bool:
    if shift == "evening":
        return True
    if day in {"Saturday", "Sunday"} and shift in {"morning", "evening"}:
        return True
    return False


def get_day_order() -> Dict[str, int]:
    return {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }


def get_sorted_days(days: List[str]) -> List[str]:
    day_order = get_day_order()
    return sorted(set(days), key=lambda d: day_order.get(d, 999))


def build_assignment_flag(
    model: cp_model.CpModel,
    vars_for_assignment: List[cp_model.IntVar],
    name: str,
) -> cp_model.IntVar:
    flag = model.NewBoolVar(name)
    model.Add(sum(vars_for_assignment) >= 1).OnlyEnforceIf(flag)
    model.Add(sum(vars_for_assignment) == 0).OnlyEnforceIf(flag.Not())
    return flag


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = get_sorted_days([r["day"] for r in requirements])
    all_shifts = sorted({r["shift"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    penalty_terms: List[cp_model.LinearExpr] = []

    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]

        if pref_type in {"shift_preference", "avoid_shift"}:
            penalty = clamp_preference_penalty(int(pref.get("penalty", 5)))
            target_day = pref.get("day")
            target_shift = pref.get("shift")

            for role in all_roles:
                for day in all_days:
                    for shift in all_shifts:
                        day_match = (target_day is None) or (day == target_day)
                        shift_match = (target_shift is None) or (shift == target_shift)

                        if day_match and shift_match:
                            key = (emp_id, day, shift, role)
                            if key in x:
                                penalty_terms.append(x[key] * penalty)

        elif pref_type in {"back_to_back_preference", "avoid_back_to_back"}:
            penalty = clamp_preference_penalty(int(pref.get("penalty", 5)))

            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = build_assignment_flag(
                    model,
                    morning_vars,
                    f"morning_assigned_{emp_id}_{day}_{i}",
                )
                evening_assigned = build_assignment_flag(
                    model,
                    evening_vars,
                    f"evening_assigned_{emp_id}_{day}_{i}",
                )
                same_day_back_to_back = model.NewBoolVar(
                    f"same_day_back_to_back_{emp_id}_{day}_{i}"
                )

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(
                    same_day_back_to_back
                )
                model.AddBoolOr(
                    [morning_assigned.Not(), evening_assigned.Not()]
                ).OnlyEnforceIf(same_day_back_to_back.Not())

                penalty_terms.append(same_day_back_to_back * penalty)

            for day_index in range(len(all_days) - 1):
                day = all_days[day_index]
                next_day = all_days[day_index + 1]

                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]
                next_morning_vars = [
                    x[(emp_id, next_day, "morning", role)]
                    for role in all_roles
                    if (emp_id, next_day, "morning", role) in x
                ]

                if not evening_vars or not next_morning_vars:
                    continue

                evening_assigned = build_assignment_flag(
                    model,
                    evening_vars,
                    f"cross_evening_assigned_{emp_id}_{day}_{i}",
                )
                next_morning_assigned = build_assignment_flag(
                    model,
                    next_morning_vars,
                    f"cross_next_morning_assigned_{emp_id}_{next_day}_{i}",
                )
                cross_day_back_to_back = model.NewBoolVar(
                    f"cross_day_back_to_back_{emp_id}_{day}_to_{next_day}_{i}"
                )

                model.AddBoolAnd(
                    [evening_assigned, next_morning_assigned]
                ).OnlyEnforceIf(cross_day_back_to_back)
                model.AddBoolOr(
                    [evening_assigned.Not(), next_morning_assigned.Not()]
                ).OnlyEnforceIf(cross_day_back_to_back.Not())

                penalty_terms.append(cross_day_back_to_back * penalty)

    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    all_penalties = penalty_terms + busy_shift_penalty_terms

    max_possible_preference_penalty = len(all_penalties) * MAX_PREFERENCE_PENALTY
    min_possible_preference_penalty = len(all_penalties) * MIN_PREFERENCE_PENALTY

    total_penalty = model.NewIntVar(
        min_possible_preference_penalty,
        max_possible_preference_penalty,
        "total_penalty",
    )

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    day_order = get_day_order()
    shift_order = {"morning": 1, "evening": 2}
    schedule.sort(
        key=lambda r: (
            day_order.get(r["day"], 999),
            shift_order.get(r["shift"], 999),
            r["role"],
            r["employee_name"],
        )
    )
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def start_next_week(
    current_data: Dict[str, Any],
    current_final_schedule: List[Dict[str, str]],
) -> Tuple[Dict[str, Any], List[Dict[str, str]]]:
    next_week_data = clone_data(current_data)
    next_week_initial_schedule = clone_schedule(current_final_schedule)
    return next_week_data, next_week_initial_schedule


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    day_order = get_day_order()
    shift_order = {
        "morning": 1,
        "evening": 2,
    }

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def get_removed_and_added_assignments(
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    old_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in old_schedule
    }
    new_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in new_schedule
    }

    removed = old_set - new_set
    added = new_set - old_set

    removed_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(removed)
    ]

    added_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(added)
    ]

    return removed_rows, added_rows


def get_back_to_back_counts(
    schedule: List[Dict[str, str]],
    employee_name: str,
) -> Tuple[int, int]:
    day_order = get_day_order()
    all_days = sorted(
        {row["day"] for row in schedule},
        key=lambda d: day_order.get(d, 999)
    )

    same_day_count = 0
    cross_day_count = 0

    for day in all_days:
        has_morning = any(
            r["employee_name"].lower() == employee_name.lower()
            and r["day"] == day
            and r["shift"] == "morning"
            for r in schedule
        )
        has_evening = any(
            r["employee_name"].lower() == employee_name.lower()
            and r["day"] == day
            and r["shift"] == "evening"
            for r in schedule
        )
        if has_morning and has_evening:
            same_day_count += 1

    for idx in range(len(all_days) - 1):
        day = all_days[idx]
        next_day = all_days[idx + 1]

        has_evening = any(
            r["employee_name"].lower() == employee_name.lower()
            and r["day"] == day
            and r["shift"] == "evening"
            for r in schedule
        )
        has_next_morning = any(
            r["employee_name"].lower() == employee_name.lower()
            and r["day"] == next_day
            and r["shift"] == "morning"
            for r in schedule
        )
        if has_evening and has_next_morning:
            cross_day_count += 1

    return same_day_count, cross_day_count


def get_shift_preference_assignment_count(
    schedule: List[Dict[str, str]],
    employee_name: str,
    target_day: str | None,
    target_shift: str | None,
) -> int:
    count = 0
    for row in schedule:
        if row["employee_name"].lower() != employee_name.lower():
            continue
        if target_day is not None and row["day"] != target_day:
            continue
        if target_shift is not None and row["shift"] != target_shift:
            continue
        count += 1
    return count


def get_unavailable_shift_assignment_count(
    schedule: List[Dict[str, str]],
    employee_name: str,
    target_day: str | None,
    target_shift: str | None,
) -> int:
    count = 0
    for row in schedule:
        if row["employee_name"].lower() != employee_name.lower():
            continue
        if target_day is not None and row["day"] != target_day:
            continue
        if target_shift is not None and row["shift"] != target_shift:
            continue
        count += 1
    return count


def generate_explanation(
    data_before: Dict[str, Any],
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> str:
    change_type = change.get("type", "").strip()
    removed_rows, added_rows = get_removed_and_added_assignments(old_schedule, new_schedule)
    lines: List[str] = []

    if change_type == "unavailable":
        employee_name = change["employee_name"]
        day = change["day"]
        shift = change["shift"]

        lines.append(
            f"{employee_name} was removed because they are unavailable for {day} {shift}."
        )

        replacement_rows = [
            row for row in added_rows
            if row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower()
        ]

        if replacement_rows:
            for row in replacement_rows:
                emp_info = next(
                    (s for s in data_before["staff"] if s["name"].lower() == row["employee_name"].lower()),
                    None
                )
                if emp_info and emp_info["employment_type"] == "full-time" and is_busy_shift(row["day"], row["shift"]):
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available, qualified as a {row['role']}, and full-time staff are preferred on busy shifts."
                    )
                else:
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available and qualified as a {row['role']}."
                    )

        if not replacement_rows:
            lines.append(
                "The schedule was re-optimized to keep the shift covered with available qualified staff."
            )

    elif change_type == "direct_swap":
        employee_name_1 = change["employee_name_1"]
        day_1 = change["day_1"]
        shift_1 = change["shift_1"]
        employee_name_2 = change["employee_name_2"]
        day_2 = change["day_2"]
        shift_2 = change["shift_2"]

        lines.append(
            f"A direct swap was performed between {employee_name_1} ({day_1} {shift_1}) and {employee_name_2} ({day_2} {shift_2})."
        )
        lines.append(
            "The swap was accepted because both employees were assigned to those shifts and remained qualified for the exchanged roles."
        )

    elif change_type in {"back_to_back_preference", "avoid_back_to_back"}:
        employee_name = change["employee_name"]
        penalty = int(change.get("penalty", 5))

        old_same_day, old_cross_day = get_back_to_back_counts(old_schedule, employee_name)
        new_same_day, new_cross_day = get_back_to_back_counts(new_schedule, employee_name)

        if penalty > 0:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to back-to-back shifts."
            )
            lines.append(
                f"Same-day back-to-back assignments (morning + evening) changed from {old_same_day} to {new_same_day}."
            )
            lines.append(
                f"Cross-day back-to-back assignments (evening + next-day morning) changed from {old_cross_day} to {new_cross_day}."
            )
            lines.append(
                "The schedule was re-optimized to reduce both same-day and cross-day back-to-back assignments when possible."
            )
        elif penalty < 0:
            lines.append(
                f"A preference was added to prefer assigning {employee_name} to back-to-back shifts."
            )
            lines.append(
                f"Same-day back-to-back assignments (morning + evening) changed from {old_same_day} to {new_same_day}."
            )
            lines.append(
                f"Cross-day back-to-back assignments (evening + next-day morning) changed from {old_cross_day} to {new_cross_day}."
            )
            lines.append(
                "The schedule was re-optimized to encourage both same-day and cross-day back-to-back assignments when possible."
            )
        else:
            lines.append(
                f"A neutral back-to-back preference was added for {employee_name} with penalty 0."
            )
            lines.append(
                f"Same-day back-to-back assignments (morning + evening) changed from {old_same_day} to {new_same_day}."
            )
            lines.append(
                f"Cross-day back-to-back assignments (evening + next-day morning) changed from {old_cross_day} to {new_cross_day}."
            )

    elif change_type == "shift_preference":
        employee_name = change["employee_name"]
        target_day = change.get("day")
        target_shift = change.get("shift")
        penalty = int(change.get("penalty", 5))

        before_count = get_shift_preference_assignment_count(
            old_schedule, employee_name, target_day, target_shift
        )
        after_count = get_shift_preference_assignment_count(
            new_schedule, employee_name, target_day, target_shift
        )

        if target_day and target_shift:
            label = f"{target_day} {target_shift}"
        elif target_day:
            label = target_day
        elif target_shift:
            label = target_shift
        else:
            label = "the selected shift pattern"

        if penalty > 0:
            lines.append(
                f"A preference was added to avoid assigning {employee_name} to {label}."
            )
            lines.append(
                f"Matching assignments changed from {before_count} to {after_count}."
            )
            lines.append(
                "The schedule was re-optimized to reduce those assignments when possible."
            )
        elif penalty < 0:
            lines.append(
                f"A preference was added to prefer assigning {employee_name} to {label}."
            )
            lines.append(
                f"Matching assignments changed from {before_count} to {after_count}."
            )
            lines.append(
                "The schedule was re-optimized to encourage those assignments when possible."
            )
        else:
            lines.append(
                f"A neutral shift preference was added for {employee_name} on {label} with penalty 0."
            )
            lines.append(
                f"Matching assignments changed from {before_count} to {after_count}."
            )

    elif change_type == "unavailable_shift":
        employee_name = change["employee_name"]
        target_day = change.get("day")
        target_shift = change.get("shift")

        before_count = get_unavailable_shift_assignment_count(
            old_schedule, employee_name, target_day, target_shift
        )
        after_count = get_unavailable_shift_assignment_count(
            new_schedule, employee_name, target_day, target_shift
        )

        if target_day and target_shift:
            label = f"{target_day} {target_shift}"
        elif target_day:
            label = target_day
        elif target_shift:
            label = target_shift
        else:
            label = "the selected unavailable pattern"

        lines.append(
            f"{employee_name} was marked as unavailable for {label}, so that pattern became a hard restriction."
        )
        lines.append(
            f"Matching assignments changed from {before_count} to {after_count}."
        )

        replacement_rows = []
        for row in added_rows:
            if row["day"].lower() != row["day"].lower():
                continue
            if target_day is not None and row["day"] != target_day:
                continue
            if target_shift is not None and row["shift"] != target_shift:
                continue
            replacement_rows.append(row)

        if replacement_rows:
            for row in replacement_rows:
                lines.append(
                    f"{row['employee_name']} was assigned because they are available and qualified as a {row['role']}."
                )

        if not replacement_rows:
            lines.append(
                "The schedule was re-optimized around this hard restriction."
            )

    else:
        lines.append("The schedule was updated and re-optimized based on the requested change.")

    full_time_staff = {
        s["name"] for s in data_before["staff"] if s["employment_type"] == "full-time"
    }

    busy_added = [row for row in added_rows if is_busy_shift(row["day"], row["shift"])]
    if busy_added:
        full_time_busy_added = [row for row in busy_added if row["employee_name"] in full_time_staff]
        if full_time_busy_added:
            names = ", ".join(sorted({row["employee_name"] for row in full_time_busy_added}))
            lines.append(
                f"Busy shifts prefer full-time staff when possible, which influenced assignments such as {names}."
            )
        else:
            lines.append(
                "Busy shifts prefer full-time staff when possible, but availability and skill constraints still had to be satisfied."
            )

    if removed_rows or added_rows:
        lines.append("Other assignments were kept unchanged when possible.")

    return "\n".join(lines)


def print_explanation(explanation: str) -> None:
    print("===== EXPLANATION =====")
    print(explanation)
    print("=======================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = clone_data(data)

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "unavailable_shift":
        employee_name = change["employee_name"].strip()
        target_day = change.get("day")
        target_shift = change.get("shift")

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability_by_pattern(
            updated_data,
            emp_id,
            available=False,
            target_day=target_day.strip() if target_day else None,
            target_shift=target_shift.strip() if target_shift else None,
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")
        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type in {"back_to_back_preference", "avoid_back_to_back"}:
        employee_name = change["employee_name"].strip()
        penalty = clamp_preference_penalty(int(change.get("penalty", 5)))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "back_to_back_preference",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type in {"shift_preference", "avoid_shift"}:
        employee_name = change["employee_name"].strip()
        target_day = change.get("day")
        target_shift = change.get("shift")
        penalty = clamp_preference_penalty(int(change.get("penalty", 5)))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        pref = {
            "type": "shift_preference",
            "employee_name": employee_name,
            "penalty": penalty,
        }
        if target_day:
            pref["day"] = target_day.strip()
        if target_shift:
            pref["shift"] = target_shift.strip()

        add_preference(updated_data, pref)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'unavailable_shift', 'direct_swap', 'back_to_back_preference', and 'shift_preference'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")
    print("3. back_to_back_preference")
    print("4. shift_preference")

    while True:
        choice = input("Enter 1, 2, 3, or 4: ").strip()
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = input("Employee 1 current day: ").strip()
        shift_1 = input("Employee 1 current shift: ").strip()

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = input("Employee 2 current day: ").strip()
        shift_2 = input("Employee 2 current shift: ").strip()

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        print("Back-to-back preference (-10 to 10):")

        while True:
            penalty_text = input("Penalty (default 5): ").strip()
            penalty = int(penalty_text) if penalty_text else 5
            try:
                penalty = clamp_preference_penalty(penalty)
                break
            except ValueError as e:
                print(e)

        return {
            "type": "back_to_back_preference",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    print("Choose shift scope:")
    print("1. a shift type (e.g. evening)")
    print("2. a day (e.g. Tuesday)")
    print("3. a specific day and shift (e.g. Tuesday evening)")

    while True:
        sub_choice = input("Enter 1, 2, or 3: ").strip()
        if sub_choice in {"1", "2", "3"}:
            break
        print("Please enter 1, 2, or 3.")

    target_day: str | None = None
    target_shift: str | None = None

    if sub_choice == "1":
        target_shift = input("Shift (e.g. evening): ").strip()
    elif sub_choice == "2":
        target_day = input("Day (e.g. Tuesday): ").strip()
    else:
        target_day = input("Day (e.g. Tuesday): ").strip()
        target_shift = input("Shift (e.g. evening): ").strip()

    hard_restriction = ask_yes_no("Hard restriction? (y/n): ")
    if hard_restriction:
        change: Dict[str, Any] = {
            "type": "unavailable_shift",
            "employee_name": employee_name,
        }
        if target_day:
            change["day"] = target_day
        if target_shift:
            change["shift"] = target_shift
        return change

    print("Shift preference (-10 to 10):")
    while True:
        penalty_text = input("Penalty (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5
        try:
            penalty = clamp_preference_penalty(penalty)
            break
        except ValueError as e:
            print(e)

    change = {
        "type": "shift_preference",
        "employee_name": employee_name,
        "penalty": penalty,
    }
    if target_day:
        change["day"] = target_day
    if target_shift:
        change["shift"] = target_shift
    return change


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule for Week 1...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    week_number = 1

    while True:
        print_schedule(schedule, title=f"WEEK {week_number} INITIAL SCHEDULE")
        print_preferences(data)

        while True:
            should_update = ask_yes_no(f"Do you want to enter an update for Week {week_number}? (y/n): ")
            if not should_update:
                break

            try:
                change_request = prompt_change_request()
                updated_schedule, updated_data = update_schedule(data, schedule, change_request)

                print_schedule(updated_schedule, title=f"WEEK {week_number} UPDATED SCHEDULE")
                compare_schedules(schedule, updated_schedule)

                explanation = generate_explanation(data, schedule, updated_schedule, change_request)
                print_explanation(explanation)

                print_preferences(updated_data)

                data = updated_data
                schedule = updated_schedule

                save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
                if save_changes:
                    save_data_to_json(data, data_file)
                    print(f"Saved updated data to {data_file}")

            except ValueError as e:
                print(f"Error: {e}")

            show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
            if show_current:
                print_schedule(schedule, title=f"WEEK {week_number} CURRENT FINAL SCHEDULE")
                print_preferences(data)

        print_schedule(schedule, title=f"WEEK {week_number} FINAL SCHEDULE")

        move_to_next_week = ask_yes_no(
            f"Do you want to move from Week {week_number} to Week {week_number + 1}? (y/n): "
        )
        if not move_to_next_week:
            print("Done.")
            break

        data, schedule = start_next_week(data, schedule)
        week_number += 1
        print(
            f"Started Week {week_number}. "
            f"By rule, Week {week_number - 1} final schedule is now Week {week_number} initial schedule."
        )


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule for Week 1...

===== WEEK 1 INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E0